# Phase 2 — LoRA Fine-Tuning of Qwen2.5-7B

**Prerequisites:** Advanced GPU tier (20 GB A100), `multilingual_pipeline.py` uploaded to this workspace.

**What this notebook does:**
1. Sets up environment and authenticates with HuggingFace
2. Imports the shared `multilingual_pipeline.py` module from Phase 1
3. Loads and formats EmpatheticDialogues into Qwen2.5 chat format
4. Loads Qwen2.5-7B-Instruct in 8-bit with LoRA adapters via PEFT
5. Trains using HuggingFace `Trainer` with gradient checkpointing
6. Evaluates a sample inference in both therapist and friend styles
7. **Saves and zips all artifacts for download before session ends**

**Session reminder:** Download `phase2_artifacts.zip` before your session terminates — the workspace is wiped.

## Section 2.1 — Environment Setup

In [1]:
# Install all required libraries
# bitsandbytes enables 8-bit model loading on the A100
# accelerate is required by the Trainer for multi-device awareness
!pip install -q transformers peft datasets torch bitsandbytes accelerate \
             scikit-learn kagglehub sentencepiece protobuf

In [2]:
import torch

# Verify CUDA is available — must show True on the A100 partition
print(f"CUDA available      : {torch.cuda.is_available()}")
print(f"GPU name            : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"VRAM total          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
print(f"PyTorch version     : {torch.__version__}")

CUDA available      : True
GPU name            : NVIDIA A100-SXM4-80GB
VRAM total          : 85.1 GB
PyTorch version     : 2.8.0+cu128


In [ ]:
# Authenticate with HuggingFace
# Paste your token here — it is only used in memory, never written to disk
# Get yours at: huggingface.co → Settings → Access Tokens → New Token (Read role)
# Also make sure you have accepted the Qwen2.5-7B-Instruct model gate at:
# https://huggingface.co/Qwen/Qwen2.5-7B-Instruct
import getpass
HF_TOKEN = getpass.getpass()  # <-- replace this

from huggingface_hub import login
login(token=HF_TOKEN)
print("HuggingFace authentication successful.")

HuggingFace authentication successful.


## Section 2.2 — Import Shared Pipeline Module

In [4]:
# multilingual_pipeline.py must be in the same directory as this notebook
# Upload it via the Jupyter file browser before running this cell
import sys, os
sys.path.insert(0, os.getcwd())

from multilingual_pipeline import (
    MultilingualPipeline,
    detect_language,
    build_chat_prompt,
    STYLES,
)

# Initialise the XLM-RoBERTa pipeline (used later for language tagging during data prep)
xlm_pipe = MultilingualPipeline()
print(xlm_pipe)

[MultilingualPipeline] Loading tokenizer: xlm-roberta-base
[MultilingualPipeline] Ready on device: cuda
MultilingualPipeline(model='xlm-roberta-base', device='cuda')


## Section 2.3 — Load and Format EmpatheticDialogues Dataset

In [5]:
import kagglehub
import pandas as pd
import os

# Download dataset — same source as Phase 1
path = kagglehub.dataset_download("atharvjairath/empathetic-dialogues-facebook-ai")
print(f"Dataset path: {path}")
print("Files:", os.listdir(path))

Dataset path: /teamspace/studios/this_studio/.cache/kagglehub/datasets/atharvjairath/empathetic-dialogues-facebook-ai/versions/1
Files: ['emotion-emotion_69k.csv']


In [6]:
# Load the CSV — column names may vary; we inspect first
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Shape: (64636, 7)
Columns: ['Unnamed: 0', 'Situation', 'emotion', 'empathetic_dialogues', 'labels', 'Unnamed: 5', 'Unnamed: 6']


,Unnamed: 0,Situation,emotion,empathetic_dialogues,labels,Unnamed: 5,Unnamed: 6
0,0,I remember going to the fireworks with my best...,sentimental,Customer :I remember going to see the firework...,"Was this a friend you were in love with, or ju...",NaN,NaN
1,1,I remember going to the fireworks with my best...,sentimental,Customer :This was a best friend. I miss her.\...,Where has she gone?,NaN,NaN
2,2,I remember going to the fireworks with my best...,sentimental,Customer :We no longer talk.\nAgent :,Oh was this something that happened because of...,NaN,NaN


In [7]:
from transformers import AutoTokenizer

# Load Qwen2.5 tokenizer — this is separate from XLM-RoBERTa
# Qwen2.5 uses its own tiktoken-based tokenizer
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
qwen_tokenizer.pad_token = qwen_tokenizer.eos_token  # Qwen has no default pad token
print(f"Qwen tokenizer loaded. Vocab size: {qwen_tokenizer.vocab_size:,}")

Qwen tokenizer loaded. Vocab size: 151,643


In [8]:
import pandas as pd
from transformers import AutoTokenizer

# Column name mapping — adjust if your CSV uses different names
# Common columns: 'Situation', 'emotion', 'empathetic_dialogues'
SITUATION_COL = "Situation"        # context / backstory
EMOTION_COL   = "emotion"          # emotion label
DIALOGUE_COL  = "empathetic_dialogues"  # the actual conversation text

MAX_SEQ_LEN = 512   # tokens per sample — safe for 20GB with batch_size=4
MAX_SAMPLES = 8000  # use a subset for the first training run; increase later

def build_training_sample(row: pd.Series, style: str = "therapist") -> dict | None:
    """
    Converts one EmpatheticDialogues row into a tokenized training sample.

    Based on inspection of `df.head()`, `DIALOGUE_COL` contains the customer's
    first turn (prefixed with 'Customer :'), and the 'labels' column contains
    the agent's response. The previous logic of splitting by '|' was incorrect.
    """
    try:
        raw_user_message = str(row[DIALOGUE_COL])
        raw_agent_response = str(row['labels']) # Assuming 'labels' contains the agent's response

        # Clean up 'Customer :' prefix from user message
        user_turn = raw_user_message.replace("Customer :", "").strip()

        # Clean up 'Agent :' prefix from agent response (if present, though 'labels' usually clean)
        target_turn = raw_agent_response.replace("Agent :", "").strip()

        if not user_turn or not target_turn:
            return None  # skip rows without a complete exchange

        situation   = str(row.get(SITUATION_COL, ""))
        emotion     = str(row.get(EMOTION_COL, ""))

        # Use detect_language to tag the user turn
        lang = detect_language(user_turn)

        # Build the Qwen2.5 chat message list via our shared pipeline function
        messages = build_chat_prompt(
            user_message=user_turn,
            style=style,
            situation=situation,
            emotion=emotion,
        )

        # Apply Qwen2.5 chat template to get the prompt string
        prompt = qwen_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        # Full text = prompt + target response + EOS
        full_text = prompt + target_turn + qwen_tokenizer.eos_token

        # Tokenize
        tokenized = qwen_tokenizer(
            full_text,
            max_length=MAX_SEQ_LEN,
            truncation=True,
            padding="max_length",
            return_tensors=None,  # return plain lists, Trainer handles batching
        )

        # Labels: mask the prompt tokens with -100 so loss is only on the response
        prompt_ids = qwen_tokenizer(prompt, return_tensors=None)["input_ids"]
        prompt_len = len(prompt_ids)
        labels = [-100] * prompt_len + tokenized["input_ids"][prompt_len:]

        # Pad or truncate labels to MAX_SEQ_LEN
        labels = labels[:MAX_SEQ_LEN]
        labels += [-100] * (MAX_SEQ_LEN - len(labels))

        tokenized["labels"] = labels
        tokenized["language"] = lang  # stored as metadata, not fed to model

        return tokenized

    except Exception as e:
        # print(f"Error processing row: {e}") # Uncomment for debugging specific row issues
        return None


print("Building training samples...")
raw_samples = df.head(MAX_SAMPLES).apply(build_training_sample, axis=1).dropna().tolist()
print(f"Valid training samples built: {len(raw_samples):,}")
print(f"Sample keys: {list(raw_samples[0].keys())}")

Building training samples...


Valid training samples built: 8,000
Sample keys: ['input_ids', 'attention_mask', 'labels', 'language']


In [9]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Remove the non-tensor 'language' metadata before passing to Dataset
tensor_keys = ["input_ids", "attention_mask", "labels"]
clean_samples = [{k: s[k] for k in tensor_keys} for s in raw_samples]

train_raw, val_raw = train_test_split(clean_samples, test_size=0.1, random_state=42)

train_dataset = Dataset.from_list(train_raw)
val_dataset   = Dataset.from_list(val_raw)

train_dataset.set_format(type="torch", columns=tensor_keys)
val_dataset.set_format(type="torch", columns=tensor_keys)

print(f"Train size : {len(train_dataset):,}")
print(f"Val size   : {len(val_dataset):,}")

Train size : 7,200
Val size   : 800


## Section 2.4 — Load Qwen2.5-7B in 8-bit and Apply LoRA

In [10]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# 8-bit quantization config — loads the 7B model into ~10GB VRAM on the A100
# Leaves ~10GB headroom for activations and gradient checkpointing
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,          # outlier threshold — default is safe
    llm_int8_has_fp16_weight=False,
)

print(f"Loading {MODEL_ID} in 8-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",          # places layers optimally across available GPU
    trust_remote_code=True,
)

# Enable gradient checkpointing to reduce activation memory
base_model.gradient_checkpointing_enable()

# Required when using gradient checkpointing with quantized models
base_model.enable_input_require_grads()

print("Base model loaded.")
print(f"VRAM used after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading Qwen/Qwen2.5-7B-Instruct in 8-bit...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Base model loaded.
VRAM used after load: 8.71 GB


In [11]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Prepare the quantized model for LoRA training
# This casts LayerNorm to fp32 and sets up gradient hooks
base_model = prepare_model_for_kbit_training(base_model)

# LoRA configuration for Qwen2.5-7B
# target_modules: the attention projection layers where adapters are injected
# r=16: adapter rank — higher = more capacity, more VRAM. 16 is a good balance on 20GB.
# lora_alpha=32: scaling factor; conventionally set to 2*r
# lora_dropout=0.05: light regularization to prevent overfitting on ~7k samples
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention projections
        "gate_proj", "up_proj", "down_proj",       # MLP / FFN projections
    ],
)

model = get_peft_model(base_model, lora_config)

# Print how many parameters are actually being trained
model.print_trainable_parameters()
# Expected output: ~0.5-1% of total params — that's the point of PEFT

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## Section 2.5 — Training

In [12]:
pip install -U transformers accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 109.1 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.4.0
    Uninstalling transformers-5.4.0:
      Successfully uninstalled transformers-5.4.0
Note: you may need to restart the kernel to use updated packages.


In [13]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

OUTPUT_DIR = "./phase2_checkpoints"

# Training hyperparameters tuned for 20GB A100 with 8-bit Qwen2.5-7B
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Batch config: effective batch size = 4 * 8 = 32
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,

    # Training duration
    num_train_epochs=3,

    # Optimizer
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,

    # Mixed precision (A100 optimized)
    bf16=True,
    fp16=False,

    # Memory optimizations
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",

    # Logging and evaluation
    logging_steps=20,
    eval_strategy="steps",   # ✅ FIXED (reverted to eval_strategy for compatibility)
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,       # ✅ important for loss

    # Misc
    report_to="none",
    dataloader_pin_memory=False,
)

# Data collator — dynamic padding
data_collator = DataCollatorForSeq2Seq(
    tokenizer=qwen_tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

# Trainer (UPDATED API)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=qwen_tokenizer,
)

print("Trainer configured. Ready to train.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer configured. Ready to train.


In [14]:
print("Starting training...")
print(f"VRAM before training: {torch.cuda.memory_allocated()/1e9:.2f} GB")

trainer.train()

print("\nTraining complete.")
print(f"VRAM after training : {torch.cuda.memory_allocated()/1e9:.2f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training...
VRAM before training: 11.06 GB


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be ca

Step,Training Loss,Validation Loss
100,0.100863,0.102018
200,0.098632,0.100292
300,0.095731,0.099665
400,0.093816,0.099119
500,0.094647,0.099012
600,0.094257,0.098989
675,0.094190,0.099008


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/auto


Training complete.
VRAM after training : 11.11 GB


## Section 2.6 — Save LoRA Adapter (NOT the full model)

In [15]:
# Best practice: save ONLY the LoRA adapter, not the full 7B base model.
#
# Why:
#   - Full merged model = ~14GB on disk. Impractical to download in session.
#   - LoRA adapter = ~300-500MB. Downloads in seconds.
#   - For Phase 4 (FastAPI), load base model once, then load adapter on top.
#     This is faster to start up and the base model can be cached on the server.
#
# The adapter folder contains:
#   adapter_config.json   — LoRA hyperparameters (r, alpha, target_modules)
#   adapter_model.bin     — the actual trained weights (~300MB)

ADAPTER_DIR = "./phase2_lora_adapter"

model.save_pretrained(ADAPTER_DIR)
qwen_tokenizer.save_pretrained(ADAPTER_DIR)  # save tokenizer alongside

print(f"LoRA adapter saved to: {ADAPTER_DIR}")

import os
for f in os.listdir(ADAPTER_DIR):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1e6
    print(f"  {f}  ({size:.1f} MB)")

LoRA adapter saved to: ./phase2_lora_adapter
  tokenizer_config.json  (0.0 MB)
  tokenizer.json  (11.4 MB)
  chat_template.jinja  (0.0 MB)
  README.md  (0.0 MB)
  adapter_config.json  (0.0 MB)
  adapter_model.safetensors  (161.5 MB)


## Section 2.7 — Sample Inference (Therapist vs Friend Style)

In [16]:
from peft import PeftModel

def generate_response(
    user_message: str,
    style: str = "therapist",
    max_new_tokens: int = 200,
) -> str:
    """
    Generates a response from the fine-tuned model.
    Uses build_chat_prompt() from multilingual_pipeline to format the input.

    Args:
        user_message   : the text input from the user
        style          : 'therapist' or 'friend'
        max_new_tokens : how long the response can be

    Returns:
        The model's response string
    """
    lang = detect_language(user_message)
    print(f"  [detected language: {lang}]")

    messages = build_chat_prompt(user_message, style=style)

    prompt = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = qwen_tokenizer(prompt, return_tensors="pt").to("cuda")

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (not the prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(new_tokens, skip_special_tokens=True)


# Test with a code-switched sentence
test_input = "I feel bahut overwhelmed these days, kuch samajh nahi aa raha"

print("=" * 60)
print(f"User: {test_input}")
print("=" * 60)

print("\n[THERAPIST STYLE]")
print(generate_response(test_input, style="therapist"))

print("\n[FRIEND STYLE]")
print(generate_response(test_input, style="friend"))

User: I feel bahut overwhelmed these days, kuch samajh nahi aa raha

[THERAPIST STYLE]
  [detected language: hi-en]


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


That is very understandable, you need some time out for yourself. Do not stress too much.

[FRIEND STYLE]
  [detected language: hi-en]
That sounds really tough! What is it you're trying to figure out?


## Section 2.8 — Package and Download All Artifacts

> **Run this cell before your session ends.** Everything in the workspace is deleted when the session terminates.

In [17]:
import shutil, os

ARTIFACT_ZIP = "phase2_artifacts"

# Collect everything needed to continue in Phase 3 and Phase 4
os.makedirs("artifact_bundle", exist_ok=True)

# 1. LoRA adapter (the trained weights)
shutil.copytree(ADAPTER_DIR, "artifact_bundle/lora_adapter", dirs_exist_ok=True)

# 2. Shared pipeline module (Phase 3 and 4 need this too)
shutil.copy("multilingual_pipeline.py", "artifact_bundle/multilingual_pipeline.py")

# 3. This notebook itself
shutil.copy("phase2_finetuning.ipynb", "artifact_bundle/phase2_finetuning.ipynb")

# Zip the bundle
shutil.make_archive(ARTIFACT_ZIP, "zip", "artifact_bundle")

zip_size = os.path.getsize(f"{ARTIFACT_ZIP}.zip") / 1e6
print(f"\nArtifact bundle ready: {ARTIFACT_ZIP}.zip  ({zip_size:.1f} MB)")
print("\nContents:")
for root, dirs, files in os.walk("artifact_bundle"):
    for f in files:
        fpath = os.path.join(root, f)
        fsize = os.path.getsize(fpath) / 1e6
        print(f"  {fpath}  ({fsize:.1f} MB)")

print("\n*** Download phase2_artifacts.zip from the Jupyter file browser NOW ***")


Artifact bundle ready: phase2_artifacts.zip  (150.7 MB)

Contents:
  artifact_bundle/multilingual_pipeline.py  (0.0 MB)
  artifact_bundle/phase2_finetuning.ipynb  (0.2 MB)
  artifact_bundle/lora_adapter/tokenizer_config.json  (0.0 MB)
  artifact_bundle/lora_adapter/tokenizer.json  (11.4 MB)
  artifact_bundle/lora_adapter/chat_template.jinja  (0.0 MB)
  artifact_bundle/lora_adapter/README.md  (0.0 MB)
  artifact_bundle/lora_adapter/adapter_config.json  (0.0 MB)
  artifact_bundle/lora_adapter/adapter_model.safetensors  (161.5 MB)

*** Download phase2_artifacts.zip from the Jupyter file browser NOW ***


## How to reload the adapter in Phase 4 (FastAPI)

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from multilingual_pipeline import build_chat_prompt, detect_language

# Load base model once at server startup
base = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-7B-Instruct',
    load_in_8bit=True, device_map='auto', trust_remote_code=True
)
# Attach your fine-tuned LoRA adapter
model = PeftModel.from_pretrained(base, './lora_adapter')
tokenizer = AutoTokenizer.from_pretrained('./lora_adapter')
```

The `multilingual_pipeline.py` module handles language detection and prompt formatting for every incoming request.